In [1]:
import torch
import torch.nn as nn
from PIL import Image
from typing import Tuple

In [17]:
from utils.config import (
    MODEL_NAME, NUM_CLASSES, DEVICE, CHECKPOINT_PATH, 
)

In [18]:
import timm 

In [19]:
def get_model(model_name: str, num_classes: int, pretrained: bool = False) -> nn.Module:
    """Instantiates the Vision Transformer (ViT) model structure from timm."""
    # Note: We set pretrained=False here because we are loading weights from a file
    model = timm.create_model(
        model_name, 
        pretrained=pretrained, 
        num_classes=num_classes
    )
    return model.to(DEVICE)


In [20]:
def load_best_model(checkpoint_path: str) -> nn.Module | None:
    """Loads the model architecture and applies the saved weights."""
    
    # 1. Instantiate the model structure
    model = get_model(
        MODEL_NAME, 
        NUM_CLASSES, 
        pretrained=False
    )
    
    # 2. Load the checkpoint file (map_location handles loading GPU model on CPU if necessary)
    try:
        checkpoint = torch.load(
            checkpoint_path, 
            map_location=DEVICE,
            weights_only=True)
    except FileNotFoundError:
        print(f"ERROR: Checkpoint file not found at {checkpoint_path}")
        return None
    
    # 3. Apply the saved weights to the model structure
    model.load_state_dict(checkpoint['model_state'])
    
    # 4. Set model to evaluation mode (crucial for disabling dropout/batch norm tracking)
    model.eval()
    
    print(f"Model ({MODEL_NAME}) successfully loaded.")
    print(f"Best Val AUC achieved during training: {checkpoint.get('best_val_auc', 'N/A')}")
    return model

In [21]:
loaded_model = load_best_model(CHECKPOINT_PATH)

Model (vit_base_patch16_224) successfully loaded.
Best Val AUC achieved during training: 1.0


In [30]:
from utils.dataset import val_tf

def check_pneumonia(model: nn.Module, image_path: str) -> Tuple[str, float]:
    """
    Predicts whether a chest X-ray contains signs of pneumonia.
    
    Returns: A tuple (Prediction Label, Confidence Score)
    """
    
    if model is None:
        return "Model Not Loaded", 0.0

    # 1. Load and Preprocess Image
    try:
        # Load and convert to RGB (as required by ViT)
        img = Image.open(image_path).convert('RGB')
        
        # Apply inference transforms
        tensor_img = val_tf(img)
        
        # Add batch dimension: [C, H, W] -> [1, C, H, W]
        input_tensor = tensor_img.unsqueeze(0).to(DEVICE)
        
    except FileNotFoundError:
        return f"Error: Image not found at {image_path}", 0.0
    except Exception as e:
        return f"Error during processing: {e}", 0.0

    # 2. Run Inference
    with torch.no_grad():
        # Set the model to evaluation mode again (safety)
        model.eval()
        
        # Use autocast for consistency, though not strictly needed for inference
        with torch.autocast(device_type=DEVICE, dtype=torch.float16): 
            logits = model(input_tensor)
        
        # Get probabilities
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
        
    # 3. Determine Prediction
    # CLASS_NAMES map: {"NORMAL": 0, "PNEUMONIA": 1}
    pneumonia_prob = probabilities[1]
    
    if pneumonia_prob > 0.88:
        prediction_label = "PNEUMONIA"
    else:
        prediction_label = "NORMAL"

    return prediction_label, float(pneumonia_prob)

In [40]:
base_path = "C:\\Users\\HP\\Desktop\\SLIIT\\Y4 SEM 1\\DL\\Ass\\Assignment\\DL-project\\ViT\\dataset\\chest_xray"

check_image_path   = base_path + "\\val\\NORMAL\\NORMAL2-IM-1436-0001.jpeg"
check_image_path_1 = base_path + "\\val\\PNEUMONIA\\person1950_bacteria_4881.jpeg" 
check_image_path_2 = base_path + "\\test\\NORMAL\\IM-0001-0001.jpeg"
check_image_path_3 = base_path + "\\test\\PNEUMONIA\\person1_virus_6.jpeg"

pred, conf = check_pneumonia(loaded_model, check_image_path_2)

# print(f"Prediction: {pred}, Confidence (Pneumonia): {conf:.4f}") 

print(f"Prediction: {pred}")
if pred == "PNEUMONIA":
    print(f"Confidence score: {conf}")

Prediction: NORMAL
